# LeNet-5 Neural Network for MNIST

### Options

In [9]:
batch_size = 20
training_epochs = 5

### Load & convert data

In [10]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

def convert_data(x_data: np.ndarray, y_data: np.ndarray):
    # hacky way to increase the size of a (unknown)x28x28 to (unknown)x32x32
    # by appending two zeros at all sides of the 28x28 matrix
    tmp = np.zeros((x_data.shape[0], 2, x_data.shape[2]))
    x_data = np.append(x_data, tmp, axis=1)
    x_data = np.append(tmp, x_data, axis=1)
    tmp = np.zeros((x_data.shape[0], x_data.shape[1], 2))
    x_data = np.append(x_data, tmp, axis=2)
    x_data = np.append(tmp, x_data, axis=2)

    # change dimensions: (unknown)x32x32 to (unknown)x32x32x1
    x_data = np.expand_dims(x_data, -1)
    # change dimensions: (unknown) to (unknown)x10
    y_data = keras.utils.to_categorical(y_data, 10)

    return (x_data, y_data)


# for now we are just interested in the training data and not the test data
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

(X_train, y_train) = convert_data(X_train, y_train)
(X_test, y_test) = convert_data(X_test, y_test)

In [11]:
print("Shapes:")
print((X_train.shape, y_train.shape))

Shapes:
((60000, 32, 32, 1), (60000, 10))


### Setup our model

In [12]:
model = keras.Sequential([
    keras.layers.Input(shape=(32, 32, 1)),
    keras.layers.Conv2D(6, kernel_size=(5, 5), activation="relu"),
    keras.layers.MaxPooling2D(pool_size=(2, 2)),
    keras.layers.Conv2D(16, kernel_size=(5, 5), activation="relu"),
    keras.layers.MaxPooling2D(pool_size=(2, 2)),
    keras.layers.Flatten(),
    keras.layers.Dense(120),
    keras.layers.Dense(84),
    keras.layers.Dense(10),
    keras.layers.Activation('softmax')
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 6)      │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 6)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 10, 10, 16)     │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 5, 5, 16)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 400)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 120)            │        48,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 84)             │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │           850 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 10)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61,706 (241.04 KB)

 Trainable params: 61,706 (241.04 KB)

 Non-trainable params: 0 (0.00 B)

### Fitting the model

In [13]:
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

hist = model.fit(X_train, y_train, batch_size=batch_size, epochs=training_epochs, validation_split=0.2)

Epoch 1/5


2024-11-06 10:36:31.743822: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 196608000 exceeds 10% of free system memory.


2400/2400 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.8577 - loss: 3.6408 - val_accuracy: 0.9627 - val_loss: 0.1310
Epoch 2/5
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.9656 - loss: 0.1194 - val_accuracy: 0.9655 - val_loss: 0.1283
Epoch 3/5
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.9660 - loss: 0.1199 - val_accuracy: 0.9747 - val_loss: 0.0965
Epoch 4/5
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.9709 - loss: 0.1041 - val_accuracy: 0.9716 - val_loss: 0.1041
Epoch 5/5
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.9723 - loss: 0.0945 - val_accuracy: 0.9736 - val_loss: 0.0919


## How good is our model for non-train data?

In [14]:
score = model.evaluate(X_test, y_test, batch_size=batch_size)

score

500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9707 - loss: 0.1029


[0.0802324116230011, 0.9761999845504761]

In [15]:
model.predict(X_train[3:5])


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step


array([[8.5534343e-11, 9.9999523e-01, 3.1975416e-09, 2.4711372e-13,
        2.0505793e-06, 2.6456890e-12, 4.1100190e-09, 2.6940736e-06,
        3.2088554e-10, 2.3010799e-08],
       [9.0088843e-09, 1.1086915e-07, 4.0661021e-09, 2.5614861e-09,
        4.1960643e-06, 1.8245383e-10, 1.1667532e-11, 1.0202648e-05,
        3.9510400e-04, 9.9959034e-01]], dtype=float32)

In [22]:
from pathlib import Path

Path("export/").mkdir(exist_ok=True)
model.save("export/original_model.keras")